# Fine-tuning Answer-Aware CrossEncoder Reranker on TechQA

**BEFORE YOU RUN**: Settings → Accelerator → GPU T4

**Improvement over the previous notebook:**

Instead of (question, doc) pairs, we use (question, answer_snippet, doc) — three separate inputs. This way, the reranker learns that a document **containing** the answer is correct, even if it doesn’t immediately resemble a troubleshooting guide.

In [ ]:
!pip install -q sentence-transformers torch

In [ ]:
import json
import random
import os
import torch
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator
from torch.utils.data import DataLoader

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU found!')

random.seed(42)

In [ ]:
BASE = '/kaggle/input/datasets/yiannisman/techqa-dataset/TechQA'

TECHNOTES_PATH = f'{BASE}/technote_corpus/full_technote_collection.sections.json'
TRAIN_QA_PATH  = f'{BASE}/training_and_dev/training_Q_A.json'
DEV_QA_PATH    = f'{BASE}/training_and_dev/dev_Q_A.json'

for path in [TECHNOTES_PATH, TRAIN_QA_PATH, DEV_QA_PATH]:
    exists = os.path.exists(path)
    status = 'CHECK' if exists else 'FAIL'
    size   = f'{os.path.getsize(path)/1024/1024:.1f} MB' if exists else 'NOT FOUND'
    print(f'{status} {os.path.basename(path)}: {size}')

In [ ]:
print('Loading Technotes...')
technotes = {}

with open(TECHNOTES_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

for doc_id, doc_data in raw.items():
    technotes[doc_id] = doc_data.get('text', '')

print(f'{len(technotes)} Loaded Technotes.')

In [ ]:
def load_qa(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

print('Loading Q&A pairs...')
train_data = load_qa(TRAIN_QA_PATH)
dev_data   = load_qa(DEV_QA_PATH)

all_qa = [
    item for item in train_data + dev_data
    if item.get('ANSWERABLE') == 'Y'
    and item.get('DOCUMENT')
    and item.get('DOCUMENT') in technotes
    and item.get('ANSWER')   # <- ANSWER field needed
    and str(item.get('ANSWER')).strip()  # <- Only if blank
]

print(f'Train questions:                {len(train_data)}')
print(f'Dev questions:                  {len(dev_data)}')
print(f'Answerable με valid doc+answer: {len(all_qa)}')

In [ ]:
# -- Creation Training Pairs ---------------------------------------------------

NUM_NEGATIVES    = 4
all_technote_ids = list(technotes.keys())

train_samples = []
skipped       = 0

for item in all_qa:
    question      = str(item.get('QUESTION_TEXT') or item.get('QUESTION_TITLE') or '').strip()
    correct_id    = item['DOCUMENT']
    correct_doc   = technotes[correct_id]

    if not question:
        skipped += 1
        continue

    # IMPORTANT: The query is ONLY the question. No answer, no HyDE.
    # We teach the model the relationship: "What the user is asking" -> "Which document solves the problem"
    pure_query = question 

    # Positive pair
    train_samples.append(InputExample(
        texts=[pure_query, correct_doc],
        label=1.0
    ))

    # Negative pairs
    neg_ids = random.sample(
        [tid for tid in all_technote_ids if tid != correct_id],
        NUM_NEGATIVES
    )
    for neg_id in neg_ids:
        train_samples.append(InputExample(
            texts=[pure_query, technotes[neg_id]],
            label=0.0
        ))

random.shuffle(train_samples)

valid_q = len(all_qa) - skipped
print(f'Total training samples: {len(train_samples)}')
print(f'  Positive pairs: {valid_q}')
print(f'  Negative pairs: {valid_q * NUM_NEGATIVES}')

In [ ]:
split_idx   = int(len(train_samples) * 0.9)
train_split = train_samples[:split_idx]
val_split   = train_samples[split_idx:]

print(f'Training samples:   {len(train_split)}')
print(f'Validation samples: {len(val_split)}')

In [ ]:
MODEL_NAME  = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
OUTPUT_PATH = '/kaggle/working/techqa-reranker-v2'
EPOCHS      = 3
BATCH_SIZE  = 16

print(f'Loading base model: {MODEL_NAME}...')
model = CrossEncoder(MODEL_NAME, num_labels=1, max_length=512)

train_dataloader = DataLoader(train_split, shuffle=True, batch_size=BATCH_SIZE)
warmup_steps     = int(len(train_dataloader) * EPOCHS * 0.1)

print(f'Training steps/epoch: {len(train_dataloader)}')
print(f'Warmup steps:         {warmup_steps}')
print(f'Start fine-tuning για {EPOCHS} epochs...')

model.fit(
    train_dataloader=train_dataloader,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    save_best_model=False
)

print(f'\nFine-tuning Completed!')

In [ ]:
print(f'Saving Model -> {OUTPUT_PATH}...')
os.makedirs(OUTPUT_PATH, exist_ok=True)
model.save(OUTPUT_PATH)

print(f'\nFiles:')
for fname in os.listdir(OUTPUT_PATH):
    size = os.path.getsize(os.path.join(OUTPUT_PATH, fname)) / (1024*1024)
    print(f'  {fname}: {size:.1f} MB')

In [ ]:
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

print('Evaluation fine-tuned model...')

val_queries = {}
for sample in val_split:
    q, doc, lbl = sample.texts[0], sample.texts[1], sample.label
    if q not in val_queries:
        val_queries[q] = {'query': q, 'positive': [], 'negative': []}
    if lbl == 1.0:
        val_queries[q]['positive'].append(doc)
    else:
        val_queries[q]['negative'].append(doc)

evaluator_samples = [
    v for v in val_queries.values()
    if len(v['positive']) > 0 and len(v['negative']) > 0
]

print(f'Evaluation queries: {len(evaluator_samples)}')
evaluator = CrossEncoderRerankingEvaluator(evaluator_samples, name='val')
scores    = evaluator(model)

print(f'\nResults:')
for key, value in scores.items():
    print(f'   {key}: {value:.4f}')
print('   (Baseline: ~0.70 | Στόχος: >0.85)')

In [ ]:
# -- Test problematic ticket DEV_Q000 --------------------------
# This was the ticket that failed with the previous model
print('Test στο DEV_Q000 (Web GUI DASH version)...')

test_question = 'You wanted to install Web GUI 8.1 FP7, but your DASH version does not meet the required version. ERROR: The installed IBM Dashboard Application Services Hub version is 3.1.0.3, but requires version 3.1.2.1 or later.'
test_answer   = 'Version 1.1.3.0 is a full refresh of Jazz for Service Management Version 1.1 Base with Modification 3.'
test_query    = f"{test_question}\n[Expected answer context]: {test_answer}"
ground_truth  = 'swg24042191'

# We take the top 20 documents from the training technotes as a proxy
# (In the actual system, they will come from BM25+embedding search)
test_doc_ids = [ground_truth, 'swg21681385', 'swg21960630', 'swg22011877',
                'swg21683704', 'swg21675814', 'swg21961858', 'swg21980863']
test_docs    = [technotes[d] for d in test_doc_ids if d in technotes]
test_ids     = [d for d in test_doc_ids if d in technotes]

pairs  = [[test_query, doc] for doc in test_docs]
scores_pred = model.predict(pairs)

ranked = sorted(zip(test_ids, scores_pred), key=lambda x: x[1], reverse=True)
print('Ranked results:')
for rank, (doc_id, score) in enumerate(ranked, 1):
    marker = ' <- CORRECT' if doc_id == ground_truth else ''
    print(f'  #{rank}: {doc_id} (score: {score:.4f}){marker}')

In [ ]:
print('='*60)
print('DONE!')
print()
print('NEXT STEPS:')
print('1. Output -> techqa-reranker-v2 -> Download')
print('2. docker exec -it knowledge_api rm -rf /app/reranker_model')
print('3. docker cp techqa-reranker-v2 knowledge_api:/app/reranker_model')
print('4. docker restart knowledge_api')
print()
print('IMPORTANT: In main.py, the query must be expanded')
print('with the answer context during the search.')
print('='*60)